# Train a model to generate coherent and contextually relevant text based on a given prompt. Starting with GPT-2, a transformer model developed by OpenAl, you will learn how to fine-tune the model on a custom dataset to create text that mimics the style and structure of your training data. 

## References:
<br>1. https://colab.research.google.com/drive/15qBZx5y9rdaQSyWpsreMDnTiZ5IlN0zD?usp=sharing
<br>2. https://huggingface.co/blog/how-to-generate

In [62]:
!pip install transformers datasets evaluate
!pip install accelerate --upgrade

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 2.8 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/10.4 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/10.4 MB 3.6 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/10.4 MB 3.6 MB/s eta 0:00:03
   -------------- ------------------------- 3.7/10.4 MB 3.6 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.4 MB 3.7 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.4 MB 3.6 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.4 MB 3.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.3/10.4 MB 3.5 MB/s eta 0:00:02
   --------------------------- ------------ 7.1/10.4 MB 3.5 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.4 MB 3.5 MB/s eta 0:00:01
   --------------------------------- ------ 8.7/10.4 MB 3.5 MB/s eta 0:00:01
   ---

In [68]:
!pip install transformers datasets torch

In [72]:
!pip install transformers datasets accelerate --upgrade
!pip install torch --upgrade
!pip install sentencepiece

   ---------------------------------------- 0.0/992.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/992.0 kB ? eta -:--:--
   --------------------- ------------------ 524.3/992.0 kB 2.1 MB/s eta 0:00:01
   ------------------------------- -------- 786.4/992.0 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 992.0/992.0 kB 1.5 MB/s eta 0:00:00


In [129]:
import warnings
warnings.filterwarnings('ignore')

In [80]:
!pip install sentencepiece

In [76]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [85]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [82]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

In [86]:
#Load tokenizer and model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

In [89]:
# Add pad token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 768)

In [95]:
#Prepare your dataset
custom_texts = [
    "Once upon a time in a land far away, there lived a brave knight.",
    "In the future, AI will revolutionize the way we work and interact.",
    "Poetry is the rhythmical creation of beauty in words."
]

In [97]:
# Convert to HuggingFace Dataset format
dataset = Dataset.from_dict({"text": custom_texts})

In [99]:
# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

In [107]:
from datasets import disable_progress_bar
disable_progress_bar()

In [109]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [111]:
# Set format for PyTorch
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

In [113]:
#DataLoader
dataloader = DataLoader(tokenized_dataset, batch_size=2, shuffle=True)

In [115]:
#Training loop (pure PyTorch)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [117]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [119]:
epochs = 3

model.train()
for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    loop = tqdm(dataloader)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch, labels=batch['input_ids'])
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(loss=loss.item())

Epoch 1


Epoch 1: 100%|██████████| 2/2 [00:14<00:00,  7.06s/it, loss=6.33]


Epoch 2


Epoch 2: 100%|██████████| 2/2 [00:11<00:00,  5.54s/it, loss=3.21]


Epoch 3


Epoch 3: 100%|██████████| 2/2 [00:12<00:00,  6.08s/it, loss=0.642]


In [121]:
#Save model
model.save_pretrained("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

('./gpt2-finetuned\\tokenizer_config.json',
 './gpt2-finetuned\\special_tokens_map.json',
 './gpt2-finetuned\\vocab.json',
 './gpt2-finetuned\\merges.txt',
 './gpt2-finetuned\\added_tokens.json')

In [123]:
#Text generation
def generate_text(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [131]:
# Example generation
prompt = "Once upon a time"
generated_text = generate_text(prompt)
print(generated_text)

Once upon a time when the world is a place of peace.
